## Módulo 1 - Sistema Baseado em Conhecimento para Gestão de Emergências

### 1. Introdução
**Contexto**

A Proteção Civil da cidade pretende um sistema de apoio à decisão para avaliar alertas ambientais em tempo real. Por exemplo, detetarpicos de calor, risco de incêndio, precipitação alta e vento forte.

**Objetivos de Aprendizagem**
1.	Reconhecer como a lógica de predicados e as regras “se-então” estruturam conhecimento;
2.	Aplicar raciocínio dedutivo e heurísticas para priorizar respostas;
3.	Compreender incerteza usando uma Rede Bayesiana simples.

**Tarefas**
1.	Definir pelo menos 10 regras de risco em lógica de predicados (“Se temperatura > 40 °C ∧ humidade < 20 % → risco_incêndio_alto”);
2.	Implementar um motor de inferência que, tendo por base um knowledge base, leia o dataset e indique ações recomendadas para cada caso presente no ficheiro.
3.	Criar uma Rede Bayesiana com 3-4 nós e usar inferência por enumeração para atualizar as probabilidades.



### 2. Configuração do Ambiente e Importação de Dados
    a. Importação de Bibliotecas

In [40]:
import sys
print(sys.executable) 
import pandas as pd
print(pd.__version__)
import json
from pathlib import Path

c:\Users\tomas\Documents\ISCTE\UCs\5.6 -  Intodução IA\IIA_Project\.venv\Scripts\python.exe
3.0.2


    b. Carregamento do Dataset

In [41]:
base_path = Path("outputs/preprocessed_for_rules.csv")
decisions_path = Path("outputs/preprocessing_decisions.csv")

if not base_path.exists():
    raise FileNotFoundError(
        "Falta outputs/preprocessed_for_rules.csv. Corre primeiro a célula final de export no eda1.ipynb."
    )

df = pd.read_csv(base_path, sep=';')
print(f"Dataset final carregado: {base_path}")
print(f"Shape: {df.shape[0]} linhas x {df.shape[1]} colunas")
display(df.head())

if decisions_path.exists():
    print("\nResumo de decisões de pré-processamento:")
    display(pd.read_csv(decisions_path).head(10))

Dataset final carregado: outputs\preprocessed_for_rules.csv
Shape: 10768 linhas x 19 colunas


,city,datetime,CO,NO2,O3,PM10,PM2.5,SO2,temperature_c,humidity_percent,wind_speed_kmh,precipitation_mm,C6H6,NOx,air_quality_good,year,month,datetime_parsed,CO_8h_avg
0,Lisboa,05/09/25 01:00,0.96,25.20,84.09,11.82,9.12,6.75,18.9,82.0,16.2,0.0,NaN,NaN,True,2025,9,2025-09-05 01:00:00,NaN
1,Lisboa,05/09/25 02:00,0.75,26.40,86.20,13.24,8.87,5.11,18.8,80.0,15.5,0.0,NaN,NaN,True,2025,9,2025-09-05 02:00:00,NaN
2,Lisboa,05/09/25 03:00,0.87,25.16,74.41,15.18,10.84,5.76,18.6,79.0,11.5,0.0,NaN,NaN,True,2025,9,2025-09-05 03:00:00,NaN
3,Lisboa,05/09/25 04:00,0.51,13.59,68.57,17.48,13.14,5.03,18.3,77.0,11.3,0.0,NaN,NaN,True,2025,9,2025-09-05 04:00:00,NaN
4,Lisboa,05/09/25 05:00,0.61,15.89,78.79,14.70,13.68,6.20,18.6,72.0,9.4,0.0,NaN,NaN,True,2025,9,2025-09-05 05:00:00,NaN



Resumo de decisões de pré-processamento:


,step,detail
0,drop_duplicates,removed=0
1,missing_threshold,threshold=75.0%
2,columns_over_threshold,"O3, PM10, PM2.5, SO2, pressure_hpa, wind_speed..."
3,columns_dropped,"pressure_hpa, wind_direction_deg, NMHC"
4,protected_features,"CO, CO_8h_avg, NO2, O3, PM10, PM2.5, SO2, air_..."
5,co_8h_avg,"rolling=8, min_periods=6, by=city"
6,outlier_method,"IQR (1.5*IQR), detection only"
7,normalization,z-score columns with std>0 (analysis copy only)


### 2.1 Verificação de Qualidade dos Dados (sanidade + outliers)

Objetivo desta secção: validar se existem valores fisicamente impossíveis e quantificar outliers antes da inferência.

Critérios de sanidade usados:
- humidade entre 0 e 100%;
- temperatura entre -30 e 60 ºC (faixa física plausível para sensores);
- vento >= 0 km/h; precipitação >= 0 mm;
- concentrações de poluentes não negativas.

Nota metodológica: outlier estatístico (IQR) não implica erro. Em contexto ambiental, extremos podem representar eventos reais de risco.

In [42]:
quality_cols = [
    "temperature_c", "humidity_percent", "wind_speed_kmh", "precipitation_mm",
    "CO", "NO2", "O3", "PM10", "PM2.5", "SO2",
    "air_quality_good", "city", "datetime"
    ]
quality_cols = [c for c in quality_cols if c in df.columns]

dq = df[quality_cols].copy()

print(f"Linhas: {len(dq)} | Colunas analisadas: {len(quality_cols)}")

# 1) Missing values
missing_pct = (dq.isna().mean() * 100).round(2).sort_values(ascending=False)
print("\nPercentagem de missing por coluna:")
display(missing_pct.to_frame("missing_pct"))

# 2) Regras de sanidade física
sanity_results = []
def add_sanity(name, mask):
    sanity_results.append({"check": name, "invalid_count": int(mask.sum()), "invalid_pct": round(100 * mask.mean(), 3)})

if "humidity_percent" in dq.columns:
    h = pd.to_numeric(dq["humidity_percent"], errors="coerce")
    add_sanity("humidity_percent < 0", h < 0)
    add_sanity("humidity_percent > 100", h > 100)

if "temperature_c" in dq.columns:
    t = pd.to_numeric(dq["temperature_c"], errors="coerce")
    add_sanity("temperature_c < -30", t < -30)
    add_sanity("temperature_c > 60", t > 60)

if "wind_speed_kmh" in dq.columns:
    w = pd.to_numeric(dq["wind_speed_kmh"], errors="coerce")
    add_sanity("wind_speed_kmh < 0", w < 0)

if "precipitation_mm" in dq.columns:
    p = pd.to_numeric(dq["precipitation_mm"], errors="coerce")
    add_sanity("precipitation_mm < 0", p < 0)

for pol in ["CO", "NO2", "O3", "PM10", "PM2.5", "SO2"]:
    if pol in dq.columns:
        x = pd.to_numeric(dq[pol], errors="coerce")
        add_sanity(f"{pol} < 0", x < 0)

sanity_df = pd.DataFrame(sanity_results)
print("\nChecks de sanidade física:")
display(sanity_df)

# 3) Outliers por IQR (apenas para visão estatística)
iqr_rows = []
for c in ["temperature_c", "humidity_percent", "wind_speed_kmh", "precipitation_mm", "CO", "NO2", "O3", "PM10", "PM2.5", "SO2"]:
    if c not in dq.columns:
        continue
    x = pd.to_numeric(dq[c], errors="coerce").dropna()
    if len(x) < 5:
        continue
    q1 = x.quantile(0.25)
    q3 = x.quantile(0.75)
    iqr = q3 - q1
    low = q1 - 1.5 * iqr
    high = q3 + 1.5 * iqr
    n_out = int(((x < low) | (x > high)).sum())
    iqr_rows.append({
        "variable": c,
        "n_non_null": int(len(x)),
        "outliers_iqr": n_out,
        "outliers_iqr_pct": round(100 * n_out / len(x), 2),
        "lower_bound": round(float(low), 3),
        "upper_bound": round(float(high), 3),
        "min": round(float(x.min()), 3),
        "p99": round(float(x.quantile(0.99)), 3),
        "max": round(float(x.max()), 3),
    })

outliers_iqr_df = pd.DataFrame(iqr_rows).sort_values("outliers_iqr_pct", ascending=False)
print("\nOutliers por IQR (interpretação: candidato a extremo, não erro automático):")
display(outliers_iqr_df)

# 4) Diagnóstico resumido para relatório
total_invalid = int(sanity_df["invalid_count"].sum()) if len(sanity_df) else 0
print("\nResumo automático:")
if total_invalid == 0:
    print("- Não foram detetados valores fisicamente impossíveis nas regras de sanidade definidas.")
else:
    print(f"- Foram detetados {total_invalid} valores fisicamente inválidos (requer limpeza/correção).")

print("- Outliers IQR devem ser avaliados com contexto de domínio; não remover automaticamente em gestão de risco.")
print("- Missing elevado em algumas variáveis pode limitar recall e cobertura de regras específicas.")

Linhas: 10768 | Colunas analisadas: 13

Percentagem de missing por coluna:


,missing_pct
wind_speed_kmh,86.61
precipitation_mm,86.61
PM2.5,86.61
PM10,86.61
O3,86.61
SO2,86.61
CO,15.34
NO2,14.96
temperature_c,3.11
humidity_percent,3.11



Checks de sanidade física:


,check,invalid_count,invalid_pct
0,humidity_percent < 0,0,0.0
1,humidity_percent > 100,0,0.0
2,temperature_c < -30,0,0.0
3,temperature_c > 60,0,0.0
4,wind_speed_kmh < 0,0,0.0
5,precipitation_mm < 0,0,0.0
6,CO < 0,0,0.0
7,NO2 < 0,0,0.0
8,O3 < 0,0,0.0
9,PM10 < 0,0,0.0



Outliers por IQR (interpretação: candidato a extremo, não erro automático):


,variable,n_non_null,outliers_iqr,outliers_iqr_pct,lower_bound,upper_bound,min,p99,max
4,CO,9116,344,3.77,-1.550,5.090,0.10,6.600,11.90
3,precipitation_mm,1442,53,3.68,0.000,0.000,0.00,0.559,15.60
9,SO2,1442,17,1.18,-2.290,14.810,0.10,14.924,18.36
5,NO2,9157,66,0.72,-58.200,249.320,2.00,242.000,340.00
7,PM10,1442,7,0.49,-7.149,49.121,1.84,45.042,58.57
0,temperature_c,10433,39,0.37,-3.950,40.450,-1.90,38.900,44.60
2,wind_speed_kmh,1442,5,0.35,-10.250,26.950,0.00,24.959,28.00
8,PM2.5,1442,1,0.07,-5.785,35.375,0.10,32.289,38.75
1,humidity_percent,10433,0,0.00,-4.400,107.600,9.20,95.000,100.00
6,O3,1442,0,0.00,19.416,118.446,28.22,107.955,114.68



Resumo automático:
- Não foram detetados valores fisicamente impossíveis nas regras de sanidade definidas.
- Outliers IQR devem ser avaliados com contexto de domínio; não remover automaticamente em gestão de risco.
- Missing elevado em algumas variáveis pode limitar recall e cobertura de regras específicas.


### 3. Definição da Base de Conhecimento (regras.json)

In [43]:
rules_path = Path("regras.json")
if not rules_path.exists():
    raise FileNotFoundError("Falta regras.json em Module_1")

with rules_path.open("r", encoding="utf-8") as f:
    rules_payload = json.load(f)

rules = rules_payload.get("rules", []) if isinstance(rules_payload, dict) else rules_payload
print(f"Total de regras: {len(rules)}")
display(pd.DataFrame([{
    "id": r.get("id"),
    "description": r.get("description"),
    "priority": r.get("priority"),
    "risk_level": (r.get("consequence") or {}).get("risk_level", r.get("risk_level"))
} for r in rules]))

Total de regras: 12


,id,description,priority,risk_level
0,R01_NO2_ALTO,Alerta NO2 critico - limite horario UE,10,ALTO
1,R02_NO2_MODERADO,Alerta NO2 preventivo,5,MODERADO
2,R03_PM10_ALTO,Particulas inalaveis PM10 excedem limite 24h,9,ALTO
3,R04_PM25_ALTO,Particulas finas PM2.5 excedem limite anual UE,10,ALTO
4,R05_O3_ALTO,Ozono troposferico - limiar de informacao,8,ALTO
5,R06_CO_ALTO,Monoxido de carbono - media 8h critica,9,ALTO
6,R07_SO2_ALTO,Dioxido de enxofre - limite horario,7,ALTO
7,R08_CALOR_EXTREMO,Onda de calor extremo,10,ALTO
8,R09_RISCO_INCENDIO,Risco maximo de incendio florestal,10,ALTO
9,R10_VENTO_FORTE,Vento forte - alerta laranja,7,ALTO


### 4. Implementação do Motor de Inferência (rules_engine.py)

In [44]:
import subprocess

rules_output = Path("outputs/rules_inference_output.csv")
cmd = [
    sys.executable,
    "rules_engine.py",
    "--input", str(base_path),
    "--rules", "regras.json",
    "--output", str(rules_output),
]

result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError("Falha na execução de rules_engine.py")

print(f"Output gerado: {rules_output}")

Processed rows: 10768
Output written to: outputs\rules_inference_output.csv
Risk distribution:
  high: 1009
  moderate: 3828
  none: 5931
Most triggered rules:
  R02_NO2_MODERADO: 4059
  R01_NO2_ALTO: 398
  R09_RISCO_INCENDIO: 388
  R04_PM25_ALTO: 140
  R12_QUALIDADE_AR_PESSIMA: 131
  R08_CALOR_EXTREMO: 61
  R03_PM10_ALTO: 7
  R11_PRECIPITACAO_INTENSA: 2

Output gerado: outputs\rules_inference_output.csv


### 5. Execução e Análise de Resultados

In [45]:
rules_df = pd.read_csv(rules_output, sep=';')
print(f"Resultados do motor de regras: {rules_df.shape[0]} linhas")

risk_dist = rules_df["overall_risk"].value_counts(dropna=False).rename_axis("overall_risk").to_frame("n")
risk_dist["pct"] = (risk_dist["n"] / len(rules_df) * 100).round(2)
display(risk_dist)

cols_to_show = [
    c for c in ["city", "datetime", "overall_risk", "matched_rule_ids", "recommended_actions"]
    if c in rules_df.columns
]
display(rules_df[cols_to_show].head(10))

Resultados do motor de regras: 10768 linhas


,n,pct
overall_risk,,
none,5931,55.08
moderate,3828,35.55
high,1009,9.37


,city,datetime,overall_risk,matched_rule_ids,recommended_actions
0,Lisboa,05/09/25 01:00,none,NaN,NaN
1,Lisboa,05/09/25 02:00,none,NaN,NaN
2,Lisboa,05/09/25 03:00,none,NaN,NaN
3,Lisboa,05/09/25 04:00,none,NaN,NaN
4,Lisboa,05/09/25 05:00,none,NaN,NaN
5,Lisboa,05/09/25 06:00,none,NaN,NaN
6,Lisboa,05/09/25 07:00,high,R04_PM25_ALTO,Grupos sensiveis permanecam em ambientes inter...
7,Lisboa,05/09/25 08:00,none,NaN,NaN
8,Lisboa,05/09/25 09:00,none,NaN,NaN
9,Lisboa,05/09/25 10:00,high,R04_PM25_ALTO,Grupos sensiveis permanecam em ambientes inter...


### 6. Modelagem de Incerteza: Rede Bayesiana (bayes_alerts.py)

In [46]:
bayes_output = Path("outputs/bayes_inference_output.csv")
cmd = [
    sys.executable,
    "bayes_alerts.py",
    "--input", str(base_path),
    "--output", str(bayes_output),
]

result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError("Falha na execução de bayes_alerts.py")

print(f"Output gerado: {bayes_output}")

Processed rows: 10768
Output written to: outputs\bayes_inference_output.csv

Output gerado: outputs\bayes_inference_output.csv


### 7. Inferência por Enumeração

In [47]:
bayes_df = pd.read_csv(bayes_output, sep=';')
print(f"Resultados Bayes: {bayes_df.shape[0]} linhas")

for col in ["p_fire_risk_true", "p_fire_risk_false"]:
    if col in bayes_df.columns:
        bayes_df[col] = pd.to_numeric(bayes_df[col], errors="coerce")

summary_bayes = bayes_df[["p_fire_risk_true", "p_fire_risk_false"]].describe().T
display(summary_bayes)

if "p_fire_risk_true" in bayes_df.columns:
    top_cols = [c for c in ["city", "datetime", "p_fire_risk_true"] if c in bayes_df.columns]
    display(bayes_df[top_cols].sort_values("p_fire_risk_true", ascending=False).head(10))

Resultados Bayes: 10768 linhas


,count,mean,std,min,25%,50%,75%,max
p_fire_risk_true,10768.0,0.230479,0.232923,0.050,0.125,0.125,0.125,0.875
p_fire_risk_false,10768.0,0.769521,0.232923,0.125,0.875,0.875,0.875,0.950


,city,datetime,p_fire_risk_true
2736,UCI_Dataset,03/05/04 16:00,0.875
2735,UCI_Dataset,03/05/04 15:00,0.875
2734,UCI_Dataset,03/05/04 14:00,0.875
3049,UCI_Dataset,16/05/04 17:00,0.875
3048,UCI_Dataset,16/05/04 16:00,0.875
3047,UCI_Dataset,16/05/04 15:00,0.875
3045,UCI_Dataset,16/05/04 13:00,0.875
3095,UCI_Dataset,18/05/04 15:00,0.875
3094,UCI_Dataset,18/05/04 14:00,0.875
3093,UCI_Dataset,18/05/04 13:00,0.875


### 7.1 Métricas Reais de Avaliação (Regras + Bayes)

Nesta secção usamos rótulo observado real (`air_quality_good`) e avaliamos:
1. Regras como classificador binário de má qualidade do ar;
2. Bayes como score probabilístico (ranking e calibração) para má qualidade do ar.

Nota crítica: o Bayes foi desenhado para risco de incêndio, por isso esta avaliação mede utilidade preditiva indireta para qualidade do ar (não equivalência semântica perfeita).

In [48]:
import numpy as np
import pandas as pd


def _confusion_counts(y_true, y_pred):
    y_true = np.asarray(y_true).astype(int)
    y_pred = np.asarray(y_pred).astype(int)
    tp = int(((y_true == 1) & (y_pred == 1)).sum())
    tn = int(((y_true == 0) & (y_pred == 0)).sum())
    fp = int(((y_true == 0) & (y_pred == 1)).sum())
    fn = int(((y_true == 1) & (y_pred == 0)).sum())
    return tn, fp, fn, tp


def _safe_div(a, b):
    return float(a / b) if b != 0 else 0.0


def _metrics_from_counts(tn, fp, fn, tp):
    total = tn + fp + fn + tp
    accuracy = _safe_div(tp + tn, total)
    precision = _safe_div(tp, tp + fp)
    recall = _safe_div(tp, tp + fn)
    f1 = _safe_div(2 * precision * recall, precision + recall)
    tnr = _safe_div(tn, tn + fp)
    balanced_acc = 0.5 * (recall + tnr)

    denom = np.sqrt((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn))
    mcc = _safe_div((tp * tn - fp * fn), denom) if denom != 0 else 0.0
    return accuracy, balanced_acc, precision, recall, f1, mcc


def _roc_auc_rank(y_true, y_score):
    y_true = np.asarray(y_true).astype(int)
    y_score = np.asarray(y_score).astype(float)
    n_pos = int((y_true == 1).sum())
    n_neg = int((y_true == 0).sum())
    if n_pos == 0 or n_neg == 0:
        return np.nan

    # Rank médio para empates
    order = np.argsort(y_score)
    ranks = np.empty_like(order, dtype=float)
    ranks[order] = np.arange(1, len(y_score) + 1)

    # Ajuste para empates: substituir por rank médio por valor
    df_rank = pd.DataFrame({"score": y_score, "rank": ranks})
    mean_ranks = df_rank.groupby("score")["rank"].transform("mean").to_numpy()

    sum_ranks_pos = mean_ranks[y_true == 1].sum()
    auroc = (sum_ranks_pos - n_pos * (n_pos + 1) / 2) / (n_pos * n_neg)
    return float(auroc)


def _auprc_step(y_true, y_score):
    y_true = np.asarray(y_true).astype(int)
    y_score = np.asarray(y_score).astype(float)
    n_pos = int((y_true == 1).sum())
    if n_pos == 0:
        return np.nan

    order = np.argsort(-y_score)
    y_true_sorted = y_true[order]

    tp_cum = np.cumsum(y_true_sorted == 1)
    fp_cum = np.cumsum(y_true_sorted == 0)

    precision = tp_cum / np.maximum(tp_cum + fp_cum, 1)
    recall = tp_cum / n_pos

    # Convenção: começa em recall=0 com precision=1
    recall_full = np.concatenate(([0.0], recall))
    precision_full = np.concatenate(([1.0], precision))

    # Área por soma de retângulos em PR step curve
    delta_recall = np.diff(recall_full)
    area = float(np.sum(delta_recall * precision_full[1:]))
    return area


def _log_loss_binary(y_true, y_score, eps=1e-15):
    y_true = np.asarray(y_true).astype(int)
    p = np.clip(np.asarray(y_score).astype(float), eps, 1 - eps)
    return float(-np.mean(y_true * np.log(p) + (1 - y_true) * np.log(1 - p)))


def _best_threshold_by_f1(y_true, y_score):
    y_true = np.asarray(y_true).astype(int)
    y_score = np.asarray(y_score).astype(float)
    thresholds = np.unique(y_score)
    if len(thresholds) == 0:
        return np.nan, np.nan

    best_f1 = -1.0
    best_t = np.nan
    for t in thresholds:
        y_pred = (y_score >= t).astype(int)
        tn, fp, fn, tp = _confusion_counts(y_true, y_pred)
        _, _, _, _, f1, _ = _metrics_from_counts(tn, fp, fn, tp)
        if f1 > best_f1:
            best_f1 = f1
            best_t = float(t)

    return best_t, float(best_f1)


def _threshold_metrics(y_true, y_score, threshold):
    y_pred = (np.asarray(y_score) >= threshold).astype(int)
    tn, fp, fn, tp = _confusion_counts(y_true, y_pred)
    acc, bal_acc, precision, recall, f1, mcc = _metrics_from_counts(tn, fp, fn, tp)
    return {
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp,
        "accuracy": acc,
        "balanced_accuracy": bal_acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "mcc": mcc,
        "predicted_positive": int(y_pred.sum()),
    }


# ========================================================
# 1) METRICAS REAIS - REGRAS vs rotulo observado
# ========================================================
rules_eval = rules_df.copy()

if {"air_quality_good", "overall_risk"}.issubset(rules_eval.columns):
    rules_eval["actual_bad_air"] = (~rules_eval["air_quality_good"].astype(bool)).astype(int)
    rules_eval["pred_bad_air"] = rules_eval["overall_risk"].isin(["moderate", "high"]).astype(int)

    y_true_r = rules_eval["actual_bad_air"].to_numpy()
    y_pred_r = rules_eval["pred_bad_air"].to_numpy()

    tn, fp, fn, tp = _confusion_counts(y_true_r, y_pred_r)
    acc, bal_acc, prec, rec, f1, mcc = _metrics_from_counts(tn, fp, fn, tp)

    cm_rules = pd.DataFrame(
        [[tn, fp], [fn, tp]],
        index=["Actual Good (0)", "Actual Bad (1)"],
        columns=["Pred Good (0)", "Pred Bad (1)"]
    )

    metrics_rules = pd.DataFrame([
        {"metric": "accuracy", "value": round(acc, 4)},
        {"metric": "balanced_accuracy", "value": round(bal_acc, 4)},
        {"metric": "precision", "value": round(prec, 4)},
        {"metric": "recall", "value": round(rec, 4)},
        {"metric": "f1", "value": round(f1, 4)},
        {"metric": "mcc", "value": round(mcc, 4)},
        {"metric": "alert_rate_pct", "value": round(100 * y_pred_r.mean(), 2)},
        {"metric": "base_rate_bad_air_pct", "value": round(100 * y_true_r.mean(), 2)},
    ])

    print("Matriz de Confusao - Regras (rotulo real air_quality_good)")
    display(cm_rules)
    print("Metricas - Regras")
    display(metrics_rules)
else:
    print("Nao foi possivel calcular metricas reais das Regras (faltam colunas).")


# ========================================================
# 2) METRICAS REAIS - BAYES (score probabilistico)
# ========================================================
bayes_eval = bayes_df.copy()

needed = {"air_quality_good", "p_fire_risk_true"}
if needed.issubset(bayes_eval.columns):
    bayes_eval["actual_bad_air"] = (~bayes_eval["air_quality_good"].astype(bool)).astype(int)
    bayes_eval["p_fire_risk_true"] = pd.to_numeric(bayes_eval["p_fire_risk_true"], errors="coerce")

    eval_mask = bayes_eval["p_fire_risk_true"].notna()
    y_true_b = bayes_eval.loc[eval_mask, "actual_bad_air"].astype(int).to_numpy()
    y_score_b = bayes_eval.loc[eval_mask, "p_fire_risk_true"].astype(float).to_numpy()

    default_threshold = 0.75
    tm = _threshold_metrics(y_true_b, y_score_b, default_threshold)
    tn_b, fp_b, fn_b, tp_b = tm["tn"], tm["fp"], tm["fn"], tm["tp"]

    auroc_b = _roc_auc_rank(y_true_b, y_score_b)
    auprc_b = _auprc_step(y_true_b, y_score_b)
    brier_b = float(np.mean((y_score_b - y_true_b) ** 2))
    ll_b = _log_loss_binary(y_true_b, y_score_b)
    best_threshold, best_f1 = _best_threshold_by_f1(y_true_b, y_score_b)

    cm_bayes = pd.DataFrame(
        [[tn_b, fp_b], [fn_b, tp_b]],
        index=["Actual Good (0)", "Actual Bad (1)"],
        columns=[f"Pred Good (0) @t={default_threshold}", f"Pred Bad (1) @t={default_threshold}"]
    )

    metrics_bayes = pd.DataFrame([
        {"metric": "accuracy@0.75", "value": round(tm["accuracy"], 4)},
        {"metric": "balanced_accuracy@0.75", "value": round(tm["balanced_accuracy"], 4)},
        {"metric": "precision@0.75", "value": round(tm["precision"], 4)},
        {"metric": "recall@0.75", "value": round(tm["recall"], 4)},
        {"metric": "f1@0.75", "value": round(tm["f1"], 4)},
        {"metric": "mcc@0.75", "value": round(tm["mcc"], 4)},
        {"metric": "auroc", "value": round(float(auroc_b), 4) if pd.notna(auroc_b) else np.nan},
        {"metric": "auprc", "value": round(float(auprc_b), 4) if pd.notna(auprc_b) else np.nan},
        {"metric": "brier", "value": round(float(brier_b), 4)},
        {"metric": "log_loss", "value": round(float(ll_b), 4)},
        {"metric": "best_f1", "value": round(float(best_f1), 4) if pd.notna(best_f1) else np.nan},
        {"metric": "best_threshold_by_f1", "value": round(float(best_threshold), 4) if pd.notna(best_threshold) else np.nan},
        {"metric": "coverage_probabilities_pct", "value": round(100 * eval_mask.mean(), 2)},
        {"metric": "base_rate_bad_air_pct", "value": round(100 * y_true_b.mean(), 2)},
    ])

    print("Matriz de Confusao - Bayes (rotulo real air_quality_good, threshold=0.75)")
    display(cm_bayes)
    print("Metricas - Bayes")
    display(metrics_bayes)

    threshold_rows = []
    for t in [0.30, 0.50, 0.60, 0.75, 0.80]:
        mt = _threshold_metrics(y_true_b, y_score_b, t)
        threshold_rows.append({
            "threshold": t,
            "precision": round(mt["precision"], 4),
            "recall": round(mt["recall"], 4),
            "f1": round(mt["f1"], 4),
            "predicted_positive": mt["predicted_positive"],
        })

    print("Sensibilidade Bayes por threshold (vs rotulo real)")
    display(pd.DataFrame(threshold_rows))
else:
    print("Nao foi possivel calcular metricas reais do Bayes (faltam colunas).")

Matriz de Confusao - Regras (rotulo real air_quality_good)


,Pred Good (0),Pred Bad (1)
Actual Good (0),3566,1009
Actual Bad (1),2365,3828


Metricas - Regras


,metric,value
0,accuracy,0.6867
1,balanced_accuracy,0.6988
2,precision,0.7914
3,recall,0.6181
4,f1,0.6941
5,mcc,0.3951
6,alert_rate_pct,44.9200
7,base_rate_bad_air_pct,57.5100


Matriz de Confusao - Bayes (rotulo real air_quality_good, threshold=0.75)


,Pred Good (0) @t=0.75,Pred Bad (1) @t=0.75
Actual Good (0),4375,200
Actual Bad (1),5505,688


Metricas - Bayes


,metric,value
0,accuracy@0.75,0.4702
1,balanced_accuracy@0.75,0.5337
2,precision@0.75,0.7748
3,recall@0.75,0.1111
4,f1@0.75,0.1943
5,mcc@0.75,0.1211
6,auroc,0.6636
7,auprc,0.7001
8,brier,0.3719
9,log_loss,1.0454


Sensibilidade Bayes por threshold (vs rotulo real)


,threshold,precision,recall,f1,predicted_positive
0,0.30,0.6693,0.2834,0.3982,2622
1,0.50,0.7787,0.1205,0.2086,958
2,0.60,0.7748,0.1111,0.1943,888
3,0.75,0.7748,0.1111,0.1943,888
4,0.80,0.7748,0.1111,0.1943,888


### 8. Conclusão

#### Leitura clara dos resultados (o que esperávamos e o que aconteceu)

**1) O que esperávamos no início**
Esperávamos que as **regras** tivessem resultados estáveis e fáceis de explicar, porque são baseadas em limites claros. Também esperávamos que o **Bayes** ajudasse a ordenar casos por nível de risco em contexto de emergência. Por fim, já sabíamos que a qualidade dos dados podia afetar muito os resultados, devido a muitos valores em falta.

**2) Regras: resultado bom, mas com limitações**
- Accuracy = **68,67%**
- Precision = **79,14%**
- Recall = **61,81%**
- F1 = **69,41%**

Isto significa, de forma simples: quando o sistema avisa que há má qualidade do ar, acerta em cerca de 8 em cada 10 vezes (bom sinal). Mas ainda deixa passar uma parte importante dos casos reais (cerca de 38%).

Porque acontece isso? A principal razão parece ser a falta de dados em variáveis importantes (por exemplo PM10, PM2.5, O3, SO2 e algumas variáveis meteorológicas). Portanto, nem tudo é problema das regras; parte do problema vem dos dados disponíveis.

**3) Bayes: útil para apoiar decisão, mas fraco para deteção direta neste formato de avaliação**
- Accuracy = **47,02%**
- Precision = **77,48%**
- Recall = **11,11%**
- F1 = **19,43%**

Em termos práticos: o modelo Bayes dá poucos alertas, e quando dá costuma acertar. No entanto, está a falhar muitos casos reais (recall muito baixo). Ou seja, neste momento está demasiado "cauteloso" para uma tarefa de prevenção quando usamos este critério de comparação.

**4) Nota metodológica importante (objetivo do módulo)**
O objetivo principal do Módulo 1 é implementar:
- um motor de inferência por regras para alertas de emergência;
- uma Rede Bayesiana simples (3-4 nós) com inferência por enumeração.

A comparação de ambos com o rótulo `air_quality_good` foi usada como **análise complementar/exploratória** para quantificar comportamento com métricas clássicas. Não deve ser lida como equivalência direta entre os dois alvos semânticos (qualidade do ar vs risco de incêndio).

**5) Sobre "números reais" nesta avaliação**
As métricas apresentadas foram calculadas com **dados observados do dataset real fornecido** (linhas reais e outputs reais dos scripts), e não com exemplos simulados. Isto reforça a utilidade prática da análise, mantendo a necessidade de interpretar os resultados no contexto correto do objetivo do módulo.

**6) O que é "alto" e "baixo" nestas métricas, em linguagem simples**
- **Precision alta**: bom, porque evita muitos alertas desnecessários.
- **Recall baixo**: problema, porque significa que casos perigosos podem ficar sem alerta.
- **F1**: resume o equilíbrio entre os dois. Nas regras está aceitável; no Bayes está baixo nesta configuração de avaliação.

**7) Conclusão final**
O sistema está funcional e foi avaliado com dados observados do dataset. As regras mostram desempenho útil e fácil de justificar para apoio à decisão. O componente Bayesiano acrescenta uma perspetiva probabilística de incerteza, mas a sua leitura quantitativa depende do alvo e do critério de avaliação escolhidos. Assim, os resultados devem ser interpretados de forma crítica e contextualizada ao objetivo de gestão de emergência do módulo.
